# LSTM & Bidirectional RNN — Beginner Notebook

This is the follow-up to the SimpleRNN sentiment analysis notebook. Here we
upgrade the same movie-review classifier in two steps:

1. **LSTM** — a smarter RNN cell with a protected long-term memory (the
   "cell state") and three gates that control what gets kept, added, or
   shown.
2. **Bidirectional** — wraps a layer so it reads the sentence **both**
   forward and backward, then combines what each direction learned.

We reuse the exact same IMDB movie review data and pipeline as before, so
we can fairly compare **SimpleRNN vs LSTM vs Bidirectional LSTM** side by
side.

Run each cell top to bottom. Every code cell has a markdown explanation
above it in simple words.

## Step 1 — Import the tools we need

Same as before, plus two new layers:
- `LSTM` — the upgraded memory cell.
- `Bidirectional` — a wrapper that runs any recurrent layer in both
  directions.

In [3]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Bidirectional, Dense

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## Step 2 — Load and prepare the data (same as Part 1)

We keep the same settings as the SimpleRNN notebook:
- Only the **10,000 most common words**.
- Every review padded/cut to **200 words**.

This keeps the comparison between models fair — the only thing changing
is the recurrent layer itself.

In [4]:
VOCAB_SIZE = 10000
MAX_LEN = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test = pad_sequences(x_test, maxlen=MAX_LEN)

print("Training reviews:", len(x_train))
print("Test reviews:", len(x_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Training reviews: 25000
Test reviews: 25000


## Step 3 — Build the LSTM model

The only change from a `SimpleRNN` model is swapping one layer:

- **`Embedding`** — same as before, turns word IDs into 32-number meaning
  vectors.
- **`LSTM(32)`** — instead of `SimpleRNN(32)`. It still outputs a 32-number
  hidden state, but internally it now keeps a separate **cell state** that
  travels across words mostly unchanged, protected by three gates:
  - **Forget gate** — decides what old information to drop.
  - **Input gate** — decides what new information to add.
  - **Output gate** — decides what to reveal as the hidden state.
- **`Dense(1, activation='sigmoid')`** — same as before, turns the final
  hidden state into a 0–1 sentiment score.

Everything else (Embedding size, Dense layer, compile settings) is
identical to the SimpleRNN model.

In [5]:
lstm_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=32, input_length=MAX_LEN),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

lstm_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 4 — Train and evaluate the LSTM model

Same training recipe as before: 5 epochs, batches of 128 reviews, 20% held
out for validation.

In [6]:
lstm_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

lstm_loss, lstm_accuracy = lstm_model.evaluate(x_test, y_test)
print(f"\nLSTM Test Accuracy: {lstm_accuracy * 100:.2f}%")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.7513 - loss: 0.5020 - val_accuracy: 0.8394 - val_loss: 0.3796
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8925 - loss: 0.2701 - val_accuracy: 0.8724 - val_loss: 0.3073
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9294 - loss: 0.1924 - val_accuracy: 0.8630 - val_loss: 0.3169
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9470 - loss: 0.1517 - val_accuracy: 0.8710 - val_loss: 0.3644
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9559 - loss: 0.1246 - val_accuracy: 0.8594 - val_loss: 0.3962
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8588 - loss: 0.4034

LSTM Test Accuracy: 85.88%


## Step 5 — Build the Bidirectional LSTM model

Now we upgrade again. `Bidirectional(LSTM(32))` runs **two** LSTMs:

- One reads the review **left to right** (forward), just like before.
- A second, separate one reads the same review **right to left**
  (backward).
- Their two 32-number outputs are **concatenated** into a single 64-number
  vector before being passed on.

This means the model sees each word with context from *both* directions —
useful for sentences where a later word (like "excellent") changes how an
earlier phrase (like "a slow start") should be read.

Notice the code change is just **one line** — wrapping `LSTM(32)` in
`Bidirectional(...)`. Everything else stays the same.

In [7]:
bilstm_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=32, input_length=MAX_LEN),
    Bidirectional(LSTM(32)),
    Dense(1, activation='sigmoid')
])

bilstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

bilstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 6 — Train and evaluate the Bidirectional LSTM model

Same recipe again, so the comparison with the plain LSTM stays fair.

In [8]:
bilstm_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

bilstm_loss, bilstm_accuracy = bilstm_model.evaluate(x_test, y_test)
print(f"\nBidirectional LSTM Test Accuracy: {bilstm_accuracy * 100:.2f}%")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.7218 - loss: 0.5356 - val_accuracy: 0.8490 - val_loss: 0.3616
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.8870 - loss: 0.2811 - val_accuracy: 0.8716 - val_loss: 0.3116
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9222 - loss: 0.2098 - val_accuracy: 0.8530 - val_loss: 0.3828
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.9438 - loss: 0.1561 - val_accuracy: 0.8758 - val_loss: 0.3264
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.9539 - loss: 0.1336 - val_accuracy: 0.8750 - val_loss: 0.3575
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8594 - loss: 0.4010

Bidirectional LSTM Test Accuracy: 85.94%


## Step 7 — Compare the two models

A quick side-by-side of test accuracy. Results can vary a little each run
(random initialization), but this gives a sense of whether the extra gates
and the extra reading direction actually helped on this dataset.

In [9]:
print("Model Comparison")
print("-" * 40)
print(f"LSTM:                {lstm_accuracy * 100:.2f}%")
print(f"Bidirectional LSTM:  {bilstm_accuracy * 100:.2f}%")

Model Comparison
----------------------------------------
LSTM:                85.88%
Bidirectional LSTM:  85.94%


## Step 8 — Try both models on our own sentences

Same encoding helper as the SimpleRNN notebook — it converts a plain
sentence into the word-ID format the models were trained on.

In [10]:
word_index = imdb.get_word_index()

def encode_review(text):
    words = text.lower().split()
    encoded = [1]  # 1 = "start of review" marker
    for word in words:
        idx = word_index.get(word, 2) + 3
        encoded.append(idx if idx < VOCAB_SIZE else 2)
    return encoded

def predict_sentiment(text, model, model_name):
    encoded = pad_sequences([encode_review(text)], maxlen=MAX_LEN)
    score = model.predict(encoded, verbose=0)[0][0]
    label = "Positive 🙂" if score > 0.5 else "Negative 🙁"
    print(f"[{model_name}] \"{text}\" -> {label} (score: {score:.2f})")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


## Step 9 — Test with a POSITIVE example

We run the same sentence through both models so we can compare their
predictions directly.

In [11]:
sentence = "this movie was absolutely wonderful and touching"
predict_sentiment(sentence, lstm_model, "LSTM")
predict_sentiment(sentence, bilstm_model, "Bidirectional LSTM")

[LSTM] "this movie was absolutely wonderful and touching" -> Positive 🙂 (score: 0.89)
[Bidirectional LSTM] "this movie was absolutely wonderful and touching" -> Positive 🙂 (score: 0.87)


## Step 10 — Test with a NEGATIVE example

In [12]:
sentence = "this movie was boring and a complete waste of time"
predict_sentiment(sentence, lstm_model, "LSTM")
predict_sentiment(sentence, bilstm_model, "Bidirectional LSTM")

[LSTM] "this movie was boring and a complete waste of time" -> Negative 🙁 (score: 0.01)
[Bidirectional LSTM] "this movie was boring and a complete waste of time" -> Negative 🙁 (score: 0.01)


## Step 11 — Test with a trickier, mixed-signal example

This is the kind of sentence Bidirectional models are meant to handle
better — a positive word arrives late, after an early negative-sounding
phrase.

In [13]:
sentence = "the movie despite a slow start was excellent"
predict_sentiment(sentence, lstm_model, "LSTM")
predict_sentiment(sentence, bilstm_model, "Bidirectional LSTM")

[LSTM] "the movie despite a slow start was excellent" -> Positive 🙂 (score: 0.76)
[Bidirectional LSTM] "the movie despite a slow start was excellent" -> Positive 🙂 (score: 0.58)


### A note on these predictions

As with the SimpleRNN notebook, don't be surprised if a short custom
sentence gets misclassified by one or both models. These models were
trained on full-length movie reviews (up to 200 words), so a 6–8 word
sentence is quite different from what they mostly learned on. This isn't a
sign the code is broken — it's a real, expected limit of a small model
trained for only 5 epochs on a fixed vocabulary.

## Step 12 — Try your own sentence

Change the text below and re-run to compare both models on your own
example.

In [15]:
sentence = "oh great, another two hours of my life I'll never get back"
predict_sentiment(sentence, lstm_model, "LSTM")
predict_sentiment(sentence, bilstm_model, "Bidirectional LSTM")

[LSTM] "oh great, another two hours of my life I'll never get back" -> Positive 🙂 (score: 0.64)
[Bidirectional LSTM] "oh great, another two hours of my life I'll never get back" -> Negative 🙁 (score: 0.26)


## Recap — what we just built

1. **Reused the data** — same IMDB reviews, padded to 200 words.
2. **Built an LSTM model** — `Embedding` → `LSTM` → `Dense`. The LSTM
   cell keeps a protected long-term "cell state," guarded by forget/input/
   output gates, so it forgets less over long text than a `SimpleRNN`.
3. **Built a Bidirectional LSTM model** — `Embedding` →
   `Bidirectional(LSTM)` → `Dense`. This reads the review both forward
   and backward and concatenates the two views before deciding.
4. **Trained and compared** both models under identical settings.
5. **Tested on real sentences** — including one designed to need context
   from both directions.

**Key takeaway:** each upgrade — `SimpleRNN` → `LSTM` →
`Bidirectional(LSTM)` — was only a one-line change in the model
definition, but each one gives the network a meaningfully different way
of handling context and memory.